# Explainability Suite Roadmap

This folder is for a step-by-step explainability study for crack segmentation.

The key principle is: **do not only draw heatmaps**. Use explainability to discover failure modes, design filters, and evaluate whether the model improves.

Recommended main comparison:

- Non-transformer main model: `Unet-efficientnet-b0-imagenet`
- Transformer reference: `Segformer-mit_b0-imagenet`
- Target settings: `Single-TB`, `Shift-TA_TC-to-TB_10pct`, `Shift-TA_TB-to-TC_10pct`

## Method 1: Failure Case Taxonomy

**Question:** What types of mistakes does the model make?

This should be the first method because it gives the rest of the XAI work a clear target.

Possible failure categories:

| Category | Meaning | Scientific signal |
|---|---|---|
| missed_thin_crack | GT crack exists but prediction misses it | high FN, low predicted probability on GT crack |
| false_positive_water_or_leaching | predicted crack overlaps other defect masks | FP overlaps original mask values 160 or 200 |
| false_positive_shadow | prediction follows dark/low-contrast regions | FP region has lower brightness or strong local contrast |
| false_positive_seam_or_edge | prediction follows long straight boundary-like structure | FP component has high elongation or strong edge response |
| fragmented_crack | prediction catches pieces but breaks continuity | many small predicted components around GT crack |
| over_thick_prediction | prediction covers crack but too wide | high TP with extra FP around crack |

This is **semi-automatic**, not fully manual. The notebook will generate candidate labels using masks, predictions, probability maps, and image statistics. You then manually audit a small number of examples to avoid overclaiming.

## Method 2: Probability / Uncertainty / Error Maps

**Question:** Where is the model confident, uncertain, correct, and wrong?

Maps to generate:

- crack probability heatmap
- entropy uncertainty map
- TP / FP / FN error map

This is model-agnostic and works for both CNN and SegFormer. It is also the cleanest bridge to post-processing filters.

## Method 3: Occlusion Sensitivity

**Question:** If we hide part of the image, does the prediction change?

This is useful because it tests causality more directly than a heatmap.

Expected interpretations:

- Occluding true cracks greatly lowers crack probability: the model relies on crack evidence.
- Occluding background shadow greatly changes prediction: the model may rely on spurious cues.
- Occluding unrelated background has little effect: good sign.

## Method 4: Multi-Level Feature Visualization

**Question:** What do early, middle, and deep layers activate on?

This is closer to the idea of looking inside CNN blocks.

Suggested visualizations:

- mean activation map per encoder stage
- top-k channel activation maps
- PCA first component of feature tensor

For crack segmentation, the expected story is:

- early layers: edges, texture, brightness changes
- middle layers: crack-like thin structures, seams, water/leaching texture
- deep layers: broader tunnel context and defect regions

## Method 5: Grad-CAM / Eigen-CAM

**Question:** Which internal feature regions support crack predictions?

This is useful for qualitative explanation, but it should not be the only method.

For segmentation, define the target carefully:

- average crack logit over predicted crack pixels
- average crack logit over ground-truth crack pixels
- average crack logit over false-positive region

If CAM highlights cracks, the model is likely using relevant evidence. If CAM highlights shadows, seams, or water/leaching regions, that suggests spurious reliance.

## Method 6: XAI-Guided Filtering

**Question:** Can explanation signals improve predictions?

Example filter signals:

- component area
- mean crack probability inside component
- peak crack probability inside component
- mean uncertainty inside component
- overlap with known non-crack masks, if available

Expected outcome:

- Precision may improve by removing false positives.
- Recall may drop if the filter removes faint true cracks.
- F1 decides whether the tradeoff is worthwhile.